<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-AIO-NSFW-EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q torchsde gguf fastapi uvicorn pydantic requests nest_asyncio

# Install ComfyUI-GGUF for loading .gguf diffusion models
!cd /content/ComfyUI/custom_nodes && git clone -q https://github.com/city96/ComfyUI-GGUF.git
!cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF && pip install -q -r requirements.txt

!apt install -qqy aria2

# Create model directories
!mkdir -p /content/ComfyUI/models/unet
!mkdir -p /content/ComfyUI/models/diffusion_models
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae

# 1. Download Qwen-Rapid-NSFW-v23 Q2_K GGUF (7.44 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Novice25/Qwen-Image-Edit-Rapid-AIO-GGUF/resolve/main/v23/Qwen-Rapid-NSFW-v23_Q2_K.gguf \
  -d /content/ComfyUI/models/unet -o Qwen-Rapid-NSFW-v23_Q2_K.gguf

# 2. Download Qwen 2.5 VL 7B FP8 Text Encoder
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors \
  -d /content/ComfyUI/models/text_encoders -o qwen_2.5_vl_7b_fp8_scaled.safetensors

# 3. Download Qwen Image VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

!ls -lh /content/ComfyUI/models/unet/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/

/content
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.

In [7]:
%cd /content/ComfyUI

import os
import sys
import gc
import torch
import logging
import warnings
import importlib.util

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

if "/content/ComfyUI" not in sys.path:
    sys.path.insert(0, "/content/ComfyUI")

# Initialize ComfyUI paths
import folder_paths
if not folder_paths.folder_names_and_paths:
    folder_paths.add_model_folder_path("unet", os.path.join(folder_paths.models_dir, "unet"))
    folder_paths.add_model_folder_path("diffusion_models", os.path.join(folder_paths.models_dir, "diffusion_models"))

# Safely load the custom nodes using ComfyUI's built-in loader mechanism
import nodes

gguf_path = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/__init__.py"
if os.path.exists(gguf_path):
    try:
        spec = importlib.util.spec_from_file_location("ComfyUI-GGUF", gguf_path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["ComfyUI-GGUF"] = module
        spec.loader.exec_module(module)
        if hasattr(module, "NODE_CLASS_MAPPINGS"):
            nodes.NODE_CLASS_MAPPINGS.update(module.NODE_CLASS_MAPPINGS)
        if hasattr(module, "NODE_DISPLAY_NAME_MAPPINGS"):
            nodes.NODE_DISPLAY_NAME_MAPPINGS.update(module.NODE_DISPLAY_NAME_MAPPINGS)
    except Exception as e:
        print(f"Failed to load ComfyUI-GGUF manually: {e}")

# Retrieve GGUF and standard loaders
UnetLoaderGGUF = nodes.NODE_CLASS_MAPPINGS.get("UnetLoaderGGUF") or nodes.NODE_CLASS_MAPPINGS.get("UnetLoaderGGUFWeb-UI")
if UnetLoaderGGUF is None:
    raise KeyError("UnetLoaderGGUF not found in NODE_CLASS_MAPPINGS.")

UnetLoaderGGUF = UnetLoaderGGUF()
CLIPLoader = nodes.NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = nodes.NODE_CLASS_MAPPINGS["VAELoader"]()

gc.collect()
torch.cuda.empty_cache()

with torch.inference_mode():
    unet = UnetLoaderGGUF.load_unet("Qwen-Rapid-NSFW-v23_Q2_K.gguf")[0]
    clip = CLIPLoader.load_clip("qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image")[0]
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]

print("Qwen-Rapid-NSFW-v23_Q2_K loaded successfully.")

/content/ComfyUI
Qwen-Rapid-NSFW-v23_Q2_K loaded successfully.


In [ ]:
%cd /content/ComfyUI

import random
import torch
import numpy as np
from PIL import Image
from IPython.display import display
from nodes import NODE_CLASS_MAPPINGS

def pil_to_comfy(pil_img):
    arr = np.array(pil_img.convert("RGB")).astype(np.float32) / 255.0
    return torch.from_numpy(arr)[None, ...]

def comfy_to_pil(comfy_img):
    arr = (comfy_img[0].detach().cpu().numpy().clip(0.0, 1.0) * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)

# Create or load a source image to edit (replace with Image.open("/content/input.png"))
input_pil = Image.new("RGB", (768, 768), (120, 140, 160))
input_tensor = pil_to_comfy(input_pil)

TextEncodeQwenImageEditPlus = NODE_CLASS_MAPPINGS["TextEncodeQwenImageEditPlus"]()
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

positive = "transform the scene into a dramatic studio portrait with rim lighting"
negative = ""

torch.cuda.empty_cache()

with torch.inference_mode():
    pos = TextEncodeQwenImageEditPlus.encode(clip, positive, vae=vae, image=input_tensor)[0]
    neg = TextEncodeQwenImageEditPlus.encode(clip, negative, vae=vae, image=input_tensor)[0]
    latent = VAEEncode.encode(vae, input_tensor)[0]

    samples = KSampler.sample(
        unet,
        random.randint(1, 2**32 - 1),
        4,          # Rapid v23 distilled steps
        1.0,        # Rapid v23 CFG
        "sa_solver",
        "beta",
        pos,
        neg,
        latent,
        1.0
    )[0]

    decoded = VAEDecode.decode(vae, samples)[0]
    out_pil = comfy_to_pil(decoded)
    display(out_pil)

/content/ComfyUI
Loading models...
Models loaded.


In [ ]:
%cd /content/ComfyUI

import os
import sys
import io
import re
import gc
import time
import base64
import random
import asyncio
import threading
import subprocess
import requests
import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import importlib.util

nest_asyncio.apply()

# Clone and install official ComfyUI-Qwen-Image node package if missing
qwen_node_dir = "/content/ComfyUI/custom_nodes/ComfyUI-Qwen-Image"
if not os.path.exists(qwen_node_dir):
    print("Cloning ComfyUI-Qwen-Image wrapper...")
    subprocess.run(["git", "clone", "-q", "https://github.com/Comfy-Org/ComfyUI-Qwen-Image.git", qwen_node_dir])
    if os.path.exists(os.path.join(qwen_node_dir, "requirements.txt")):
        subprocess.run(["pip", "install", "-q", "-r", os.path.join(qwen_node_dir, "requirements.txt")])

if qwen_node_dir not in sys.path:
    sys.path.insert(0, qwen_node_dir)

import nodes
from nodes import NODE_CLASS_MAPPINGS

# Recursively find and load modules defining Qwen encoding nodes
loaded = False
for root, dirs, files in os.walk(qwen_node_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            module_name = file[:-3]
            try:
                spec = importlib.util.spec_from_file_location(module_name, file_path)
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)
                if hasattr(module, "NODE_CLASS_MAPPINGS"):
                    NODE_CLASS_MAPPINGS.update(module.NODE_CLASS_MAPPINGS)
                    loaded = True
            except Exception:
                pass

# If it still isn't loaded, let's create a custom TextEncodeQwen2_5_VL node manually mapping directly to ComfyUI's standard CLIP loader
if "TextEncodeQwen2_5_VL" not in NODE_CLASS_MAPPINGS and "TextEncodeQwenImageEditPlus" not in NODE_CLASS_MAPPINGS:
    print("Falling back to manual registration of Qwen Text Encoder nodes...")
    class TextEncodeQwenManual:
        @classmethod
        def INPUT_TYPES(s):
            return {
                "required": {
                    "clip": ("CLIP",),
                    "text": ("STRING", {"multiline": True}),
                    "image": ("IMAGE",),
                }
            }
        RETURN_TYPES = ("CONDITIONING",)
        FUNCTION = "encode"
        CATEGORY = "conditioning"

        def encode(self, clip, text, image, vae=None):
            # Map using the built-in clip encode wrapper
            tokens = clip.tokenize(text)
            cond, pooled = clip.encode_from_tokens(tokens, return_pooled=True)
            return ([[cond, {"pooled_output": pooled}]],)

    NODE_CLASS_MAPPINGS["TextEncodeQwenImageEditPlus"] = TextEncodeQwenManual
    NODE_CLASS_MAPPINGS["TextEncodeQwen2_5_VL"] = TextEncodeQwenManual

TextEncodeQwenImageEditPlus = NODE_CLASS_MAPPINGS.get("TextEncodeQwenImageEditPlus") or NODE_CLASS_MAPPINGS.get("TextEncodeQwen2_5_VL")
TextEncodeQwenImageEditPlus = TextEncodeQwenImageEditPlus()
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

gpu_lock = asyncio.Lock()

class GenerateRequest(BaseModel):
    positive: str = ""
    negative: str = ""
    steps: int = 4
    cfg: float = 1.0
    width: int = 768
    height: int = 768
    seed: int = -1
    sampler: str = "sa_solver"
    scheduler: str = "beta"
    denoise: float = 1.0
    loras: list = []
    reference_image: str | None = None

def decode_base64_image(b64_str: str | None, fallback_size=(768, 768)) -> Image.Image:
    if not b64_str:
        return Image.new("RGB", fallback_size, (128, 128, 128))
    cleaned = re.sub(r"^data:image/\w+;base64,", "", b64_str.strip())
    raw = base64.b64decode(cleaned)
    return Image.open(io.BytesIO(raw)).convert("RGB")

def pil_to_comfy(pil_img: Image.Image) -> torch.Tensor:
    arr = np.array(pil_img).astype(np.float32) / 255.0
    return torch.from_numpy(arr)[None, ...]

def comfy_to_pil(comfy_img: torch.Tensor) -> Image.Image:
    arr = (comfy_img[0].detach().cpu().numpy().clip(0.0, 1.0) * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)

@app.get("/user_models")
def get_user_models():
    return {
        "checkpoints": ["Qwen-Rapid-NSFW-v23_Q2_K.gguf"],
        "loras": []
    }

@app.post("/generate")
async def generate(req: GenerateRequest):
    async with gpu_lock:
        try:
            seed = req.seed if req.seed > 0 else random.randint(1, 2**32 - 1)
            target_w = min(max((req.width // 16) * 16, 256), 1024)
            target_h = min(max((req.height // 16) * 16, 256), 1024)

            print(f"Generating: {req.steps} steps, cfg={req.cfg}, {target_w}x{target_h}, seed={seed}")

            gc.collect()
            torch.cuda.empty_cache()

            input_pil = decode_base64_image(req.reference_image, (target_w, target_h))
            if input_pil.size != (target_w, target_h):
                input_pil = input_pil.resize((target_w, target_h), Image.Resampling.LANCZOS)
            input_tensor = pil_to_comfy(input_pil)

            with torch.inference_mode():
                pos = TextEncodeQwenImageEditPlus.encode(clip, req.positive, vae=vae, image=input_tensor)[0]
                neg = TextEncodeQwenImageEditPlus.encode(clip, req.negative, vae=vae, image=input_tensor)[0]
                latent = VAEEncode.encode(vae, input_tensor)[0]

                samples = KSampler.sample(
                    unet,
                    seed,
                    req.steps,
                    req.cfg,
                    req.sampler,
                    req.scheduler,
                    pos,
                    neg,
                    latent,
                    req.denoise
                )[0]

                decoded = VAEDecode.decode(vae, samples)[0]

            out_pil = comfy_to_pil(decoded)
            buf = io.BytesIO()
            out_pil.save(buf, format="PNG")

            gc.collect()
            torch.cuda.empty_cache()

            return {"image": base64.b64encode(buf.getvalue()).decode("utf-8"), "seed": seed}
        except Exception as e:
            gc.collect()
            torch.cuda.empty_cache()
            import traceback
            traceback.print_exc()
            return {"error": str(e)}

@app.get("/")
def root():
    return {"status": "ok", "model": "Qwen-Rapid-NSFW-v23_Q2_K.gguf"}

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# Cleanup existing active ports and services
!fuser -k 8000/tcp >/dev/null 2>&1 || true
!pkill -f cloudflared >/dev/null 2>&1 || true
time.sleep(2)

# Install cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing Cloudflare tunnel...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Start API Server in daemon thread
threading.Thread(target=start_server, daemon=True).start()
time.sleep(4)

# Start Cloudflare Tunnel and parse public URL
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting tunnel, waiting for address...")
url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print("=" * 50)
    print(f"READY: {url}")
    print("=" * 50)
else:
    print("Failed to retrieve Cloudflare tunnel URL.")

try:
    while True:
        time.sleep(30)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    cf_proc.terminate()
    print("\nStopped.")

/content/ComfyUI
Cloning ComfyUI-Qwen-Image wrapper...
Falling back to manual registration of Qwen Text Encoder nodes...
^C
Installing Cloudflare tunnel...
Starting tunnel, waiting for address...
READY: https://residents-definition-macintosh-soundtrack.trycloudflare.com
..Generating: 20 steps, cfg=4.0, 1024x1024, seed=2808556389
.

Traceback (most recent call last):
  File "/tmp/ipykernel_5989/2437282958.py", line 157, in generate
    pos = TextEncodeQwenImageEditPlus.encode(clip, req.positive, vae=vae, image=input_tensor)[0]
                                             ^^^^
NameError: name 'clip' is not defined


..